# 🥒 Zucchini Color Scoring — LAB Winners Deep Ablation: CSVS01, Partial Fine-Tuning, Confusion Diagnostics

This notebook is a **focused, research-level deep-dive** on the two configurations that surpassed
**R² > 0.90** in `zucchini_transfer_learning_v2.ipynb`'s LAB color-space ablation:

| Backbone | Color space | Filter (in v2) | v2 OOF R² |
|---|---|---|---|
| **ResNet-50**     | LAB | Statistical (K=1.5σ) | 0.9006 |
| **ConvNeXt-Tiny** | LAB | Statistical (K=1.5σ) | 0.9060 (0.9078 w/ Z-score) |

Rather than assume LAB + Statistical still wins, this notebook **re-runs the full color-space
(RGB/HSV/LAB/XYZ) and filtering-method (No-Filter/Statistical/Z-score) ablation from scratch** on a
new dataset, then goes further than v2 in three ways:

1. **New dataset** — `CSVS01`, entirely replacing the old `NOCC_ROOT` source.
2. **Partial fine-tuning** — the last block of each backbone is unfrozen and fine-tuned
   (v2 backbones were 100% frozen). Fine-tuning is repeated **independently inside every CV fold**
   so no validation image ever influences the weights used to embed it (see Section 15).
3. **Confusion diagnostics** — a rounded-score confusion matrix, plus an image "confusion frieze"
   (one row per true score, up to 10 example crops per row, ordered by predicted score) for direct
   visual inspection of where and how each model errs.

**Only ResNet-50 and ConvNeXt-Tiny are considered** — no other backbones, and no CNN-from-scratch
comparison. Dataset, filtering code, handcrafted color-space features, and the CV/XGBoost trainer
are ported verbatim from `zucchini_transfer_learning_v2.ipynb` / `zucchini_ml_visulisation_ranking_v3.ipynb`
so results stay comparable to prior notebooks in this project.

⚠ **This notebook must be run on a machine with access to the `CSVS01` path below** (a Windows
network/server path) and, for practical fine-tuning runtimes, a CUDA GPU. It has not been executed
in this environment.

## 0 · Imports & Setup

In [ ]:
import os, glob, random, warnings, time, hashlib, json as _json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import tifffile as tiff
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

from scipy.stats import spearmanr, kendalltau, skew, kurtosis

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.models import (resnet50, ResNet50_Weights,
                                 convnext_tiny, ConvNeXt_Tiny_Weights)
from torch.utils.data import Dataset, DataLoader

import xgboost as xgb
import umap

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch device : {device}')
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
XGB_DEVICE = device.type   # 'cuda' or 'cpu' — passed straight to XGBoost's `device` param

print(f'torch        : {torch.__version__}')
print(f'torchvision  : {torchvision.__version__}')
print(f'xgboost      : {xgb.__version__}')

if device.type != 'cuda':
    print('\n⚠ No CUDA GPU detected. Section 16 (per-fold partial fine-tuning: 2 backbones x 5 '
          'folds = 10 fine-tune runs) will be very slow on CPU. Strongly recommended to run this '
          'notebook on a GPU machine.')

# ── Corporate-network SSL workaround ────────────────────────────────────
# Flip to True if torchvision.models.*(weights=...) fails with SSLCertVerificationError
# when fetching pretrained weights from download.pytorch.org.
CORPORATE_NETWORK_SSL_BYPASS = False
if CORPORATE_NETWORK_SSL_BYPASS:
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
    print('⚠ SSL certificate verification disabled for weight downloads.')

## 1 · Configuration

In [ ]:
# ── EDIT THIS PATH if it moves — CSVS01, replacing v2's NOCC_ROOT entirely ──
NOCC_ROOT = r"F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\nocc\CSVS01"
# ─────────────────────────────────────────────────────────────────────────

IMG_SIZE       = (128, 128)     # native load size — identical to prior notebooks
BACKBONE_SIZE  = 128            # ImageNet-pretrained backbones expect 224x224 input
BATCH_SIZE     = 32             # batch size for frozen-embedding extraction (Section 7)
K_FOLDS        = 5              # confirmed: 5-fold CV (not increased)
SEED           = 42
FG_THRESH      = 10             # foreground threshold — same convention as v3 (pixel > 10 on any channel)
K_STAT         = 1.5            # sigma half-width for the statistical LAB filter (v3 default)
Z_THRESH       = 2.0            # |z| threshold for the z-score filter (v3 default)
N_HIST_BINS    = 8              # histogram bins per channel for handcrafted features

SCORE_FOLDERS = list(range(1, 10))       # folders 1-9 -> scores 1-9 (matches "Score=01" ... "Score=09")
NUM_CLASSES   = 9
IDX_TO_SCORE  = np.arange(1, 10, dtype=np.float32)   # 1..9

# ── Partial fine-tuning hyperparameters (Section 15-16) ────────────────────
FT_EPOCHS        = 15      # max epochs per fold's fine-tune run (early stopping usually stops sooner)
FT_PATIENCE       = 5       # early-stopping patience on the inner validation split
FT_BATCH_SIZE      = 16
FT_LR_BACKBONE     = 1e-5   # small LR for the unfrozen backbone block (discriminative LR)
FT_LR_HEAD         = 1e-3   # larger LR for the freshly-initialized regression head
FT_WEIGHT_DECAY    = 1e-4
FT_INNER_VAL_FRAC  = 0.15   # fraction of each fold's TRAIN split carved out for fine-tune early stopping
                            # (never touches that fold's held-out validation images)

# ── Confusion frieze ─────────────────────────────────────────────────────
FRIEZE_MAX_PER_ROW = 10     # max images per true-score row

CACHE_DIR = 'lab_winners_cache'
OUT_DIR   = 'lab_winners_outputs'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

FORCE_RECOMPUTE_EMBEDDINGS = False   # set True to ignore cached .npy embeddings/features

def savefig(name, dpi=150):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  💾 {path}')

print(f'NOCC_ROOT (CSVS01)  : {NOCC_ROOT}')
print(f'Image load size     : {IMG_SIZE}')
print(f'Backbone input size : {BACKBONE_SIZE}x{BACKBONE_SIZE}')
print(f'K-folds             : {K_FOLDS}')
print(f'Fine-tune epochs    : {FT_EPOCHS} (patience={FT_PATIENCE})')
print(f'Frieze max/row      : {FRIEZE_MAX_PER_ROW}')
print(f'Cache dir           : {CACHE_DIR}/')

## 2 · Load Images (nocc, CSVS01) — same loader as prior notebooks

`load_tif_rgb` and `collect_data` are ported **verbatim** from `zucchini_transfer_learning_v2.ipynb`
so folder-naming quirks are handled identically. `collect_data` already tries `Score=01`, `Score=1`,
`01`, and `1` for each score folder — matching the `Score=01` ... `Score=09` naming under CSVS01, so
no changes were needed here beyond repointing `NOCC_ROOT`.

In [ ]:
def load_tif_rgb(path, size):
    img = tiff.imread(path)
    if img.ndim == 4: img = img[0]
    if img.ndim == 3 and img.shape[0] in (1, 3, 4):
        img = np.transpose(img, (1, 2, 0))
    if img.ndim == 2:        img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1: img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4: img = img[..., :3]
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        lo, hi = img.min(), img.max()
        img = (img - lo) / (hi - lo + 1e-8)
        img = (img * 255).astype(np.uint8)
    return cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_AREA)


def rgb_to_lab(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)


def collect_data(root, score_folders, img_size):
    '''Walk each Score=NN (or plain NN) subfolder under root and load nocc TIFFs.'''
    rgb_list, lab_list, label_list, path_list = [], [], [], []

    print(f'\nSearching root: {root}')
    if not os.path.exists(root):
        raise FileNotFoundError(f'NOCC_ROOT not found: {root}')

    top_items = sorted(os.listdir(root))
    print(f'  Contents ({len(top_items)} items): {top_items[:20]}')

    for folder_num in score_folders:
        candidates = [
            os.path.join(root, str(folder_num)),
            os.path.join(root, f'Score={folder_num:02d}'),
            os.path.join(root, f'Score={folder_num}'),
            os.path.join(root, f'{folder_num:02d}'),
        ]
        folder_path = next((p for p in candidates if os.path.isdir(p)), None)
        if folder_path is None:
            print(f'  [Score {folder_num}] ⚠  No matching folder found')
            continue

        all_tifs = (glob.glob(os.path.join(folder_path, '*.tif'))
                  + glob.glob(os.path.join(folder_path, '*.TIF'))
                  + glob.glob(os.path.join(folder_path, '*.tiff')))
        all_tifs = list(set(all_tifs))
        nocc_tifs = [p for p in all_tifs if 'nocc' in os.path.basename(p).lower()]
        chosen = nocc_tifs if nocc_tifs else all_tifs
        tag = 'nocc' if nocc_tifs else 'ALL (no nocc found — using fallback)'
        print(f'  [Score {folder_num}] folder={os.path.basename(folder_path)} '
              f'| all_tifs={len(all_tifs)} | {tag}={len(chosen)}')

        for p in chosen:
            try:
                rgb = load_tif_rgb(p, img_size)
                lab = rgb_to_lab(rgb)
                rgb_list.append(rgb)
                lab_list.append(lab)
                label_list.append(folder_num - 1)   # 0-indexed
                path_list.append(p)
            except Exception as e:
                print(f'    ⚠ Failed {os.path.basename(p)}: {e}')

    if len(rgb_list) == 0:
        raise RuntimeError(f'No images loaded! Check NOCC_ROOT: {root}')

    print(f'\n  ✔ Total loaded: {len(rgb_list)} images across {len(set(label_list))} score classes\n')
    return (np.array(rgb_list, dtype=np.uint8),
            np.array(lab_list, dtype=np.uint8),
            np.array(label_list, dtype=np.int32),
            path_list)


X_rgb, X_lab, y_raw, img_paths = collect_data(NOCC_ROOT, SCORE_FOLDERS, IMG_SIZE)
y = IDX_TO_SCORE[y_raw]     # continuous regression target, 1..9

unique, counts = np.unique(y_raw, return_counts=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(unique + 1, counts, color='steelblue', edgecolor='black')
ax.set_xlabel('Score'); ax.set_ylabel('Count'); ax.set_title('CSVS01 — Image Distribution per Score Class')
ax.set_xticks(unique + 1)
for u, c in zip(unique, counts):
    ax.text(u + 1, c + 0.3, str(c), ha='center', fontsize=9)
plt.tight_layout(); savefig('00_class_distribution.png'); plt.show()

print(f'X_rgb : {X_rgb.shape}   y : {y.shape}   score range: [{y.min():.0f}, {y.max():.0f}]')

## 3 · Filtering — No-Filter vs Statistical vs Z-score (ported from v3 / v2)

Same three pixel-filtering strategies as prior notebooks, applied to the CSVS01 nocc-cropped images.
All three are computed and cached here (not just the primary one) so **Section 14 can re-run the
full filtering-method ablation** rather than assuming Statistical (K=1.5σ) still wins on the new
dataset.

In [ ]:
def stat_filter(img_rgb, k=K_STAT):
    '''Statistical LAB interval filter — keep pixels within [mean +/- k*sigma] per LAB channel.'''
    H, W = img_rgb.shape[:2]
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[..., 0] * 100.0 / 255.0
    a_img = lab[..., 1] - 128.0
    b_img = lab[..., 2] - 128.0
    coords = np.argwhere(fg_mask)
    if len(coords) == 0:
        return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}
    r, c = coords[:, 0], coords[:, 1]
    keep_px = np.ones(len(coords), dtype=bool)
    for ch_img in [L_img, a_img, b_img]:
        vals = ch_img[r, c]
        mu, sd = vals.mean(), vals.std()
        keep_px &= (vals >= mu - k * sd) & (vals <= mu + k * sd)
    kept_mask = np.zeros((H, W), dtype=bool)
    kept_mask[r[keep_px], c[keep_px]] = True
    out = np.full_like(img_rgb, 255)
    out[kept_mask] = img_rgb[kept_mask]
    return {'out': out, 'kept_mask': kept_mask, 'kept_pct': float(keep_px.sum() / len(coords) * 100)}


def zscore_filter(img_rgb, z_thresh=Z_THRESH):
    '''Z-score filter — keep pixels with |z| < z_thresh for all LAB channels.'''
    H, W = img_rgb.shape[:2]
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[..., 0] * 100.0 / 255.0
    a_img = lab[..., 1] - 128.0
    b_img = lab[..., 2] - 128.0
    coords = np.argwhere(fg_mask)
    if len(coords) == 0:
        return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}
    r, c = coords[:, 0], coords[:, 1]
    keep_px = np.ones(len(coords), dtype=bool)
    for ch_img in [L_img, a_img, b_img]:
        vals = ch_img[r, c]
        mu, sigma = vals.mean(), vals.std()
        if sigma > 1e-6:
            z = np.abs((vals - mu) / sigma)
            keep_px &= (z < z_thresh)
    kept_mask = np.zeros((H, W), dtype=bool)
    kept_mask[r[keep_px], c[keep_px]] = True
    out = np.full_like(img_rgb, 255)
    out[kept_mask] = img_rgb[kept_mask]
    return {'out': out, 'kept_mask': kept_mask, 'kept_pct': float(keep_px.sum() / len(coords) * 100)}


def no_filter(img_rgb):
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}


FILTER_METHODS = {
    'No Filter':             no_filter,
    'Statistical (K=1.5σ)':  stat_filter,
    'Z-score (|z|<2)':       zscore_filter,
}
PRIMARY_FILTER = 'Statistical (K=1.5σ)'   # used to build the Section-4 primary feature set;
                                          # Section 14 re-benchmarks all three per backbone

filter_outputs = {}
for fname, ffunc in FILTER_METHODS.items():
    t0 = time.time()
    res = [ffunc(img) for img in X_rgb]
    filter_outputs[fname] = {
        'images':    np.array([r['out'] for r in res], dtype=np.uint8),
        'masks':     np.array([r['kept_mask'] for r in res], dtype=bool),
        'kept_pcts': np.array([r['kept_pct'] for r in res], dtype=np.float32),
    }
    pcts = filter_outputs[fname]['kept_pcts']
    print(f'  {fname:24s} kept mean={pcts.mean():5.1f}%  std={pcts.std():4.1f}%  '
          f'({time.time()-t0:.1f}s)')

X_rgb_filtered = filter_outputs[PRIMARY_FILTER]['images']
X_masks        = filter_outputs[PRIMARY_FILTER]['masks']
print(f'\nPrimary filter for the Section-4 feature set: {PRIMARY_FILTER}')

In [ ]:
fig, axes = plt.subplots(1, len(FILTER_METHODS) + 1, figsize=(4 * (len(FILTER_METHODS) + 1), 4))
sample_idx = int(np.argmax(y_raw == y_raw[0]))
axes[0].imshow(X_rgb[sample_idx]); axes[0].set_title('Original (nocc)'); axes[0].axis('off')
for ax, fname in zip(axes[1:], FILTER_METHODS):
    ax.imshow(filter_outputs[fname]['images'][sample_idx])
    ax.set_title(f"{fname}\nkept={filter_outputs[fname]['kept_pcts'][sample_idx]:.0f}%")
    ax.axis('off')
plt.suptitle('CSVS01 — Filtering Method Comparison — Example Image', y=1.05)
plt.tight_layout(); savefig('01_filter_comparison_example.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
box_data = [filter_outputs[f]['kept_pcts'] for f in FILTER_METHODS]
ax.boxplot(box_data, labels=list(FILTER_METHODS.keys()))
ax.set_ylabel('Kept foreground pixels (%)'); ax.set_title('CSVS01 — Kept-Pixel % by Filtering Method')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); savefig('02_filter_kept_pct_boxplot.png'); plt.show()

## 4 · Multi-Color-Space Handcrafted Features — RGB, HSV, LAB, XYZ (ported from v3/v2)

Per channel, per color space: **mean, std, skewness, kurtosis, range, IQR, p10, p90** +
an `N_HIST_BINS`-bin histogram, computed on the primary-filtered images. Each color space is kept
**individually** (not just concatenated) so Section 13 can re-run the per-color-space ablation.

In [ ]:
COLOR_SPACES = ['RGB', 'HSV', 'LAB', 'XYZ']
CS_CHANNEL_NAMES = {'RGB': ['R', 'G', 'B'], 'HSV': ['H', 'S', 'V'],
                     'LAB': ['L', 'a', 'b'], 'XYZ': ['X', 'Y', 'Z']}


def convert_color_space(img_rgb, space):
    if space == 'RGB': return img_rgb.copy()
    if space == 'HSV': return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    if space == 'LAB': return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    if space == 'XYZ': return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2XYZ)
    raise ValueError(f'Unknown color space: {space}')


def extract_features_single_space(img_rgb, fg_mask, space, n_bins=N_HIST_BINS):
    '''8 stats (mean, std, skew, kurtosis, range, IQR, p10, p90) + n_bins histogram, per channel.'''
    img_cs = convert_color_space(img_rgb, space)
    feats = []
    for c in range(3):
        vals = img_cs[:, :, c][fg_mask].astype(np.float32)
        if len(vals) == 0:
            feats.extend([0.0] * (8 + n_bins))
            continue
        feats.append(float(vals.mean()))
        feats.append(float(vals.std()))
        feats.append(float(skew(vals)))
        feats.append(float(kurtosis(vals)))
        p10, p25, p75, p90 = np.percentile(vals, [10, 25, 75, 90])
        feats.append(float(vals.max() - vals.min()))
        feats.append(float(p75 - p25))
        feats.append(float(p10))
        feats.append(float(p90))
        hist, _ = np.histogram(vals, bins=n_bins, range=(0, 256), density=False)
        hist = hist.astype(np.float32); hist /= (hist.sum() + 1e-8)
        feats.extend(hist.tolist())
    return np.array(feats, dtype=np.float32)


def build_feature_names(space, n_bins=N_HIST_BINS):
    names = []
    for ch in CS_CHANNEL_NAMES[space]:
        names += [f'{space}_{ch}_mean', f'{space}_{ch}_std', f'{space}_{ch}_skew', f'{space}_{ch}_kurt',
                  f'{space}_{ch}_range', f'{space}_{ch}_iqr', f'{space}_{ch}_p10', f'{space}_{ch}_p90']
        names += [f'{space}_{ch}_bin{b}' for b in range(n_bins)]
    return names


def extract_multi_cs_features_for_dataset(images, masks, cache_tag):
    '''Extract per-CS + Combined handcrafted features for a full image set, with disk caching.'''
    cache_path = os.path.join(CACHE_DIR, f'handcrafted_{cache_tag}.npz')
    if os.path.exists(cache_path) and not FORCE_RECOMPUTE_EMBEDDINGS:
        npz = np.load(cache_path)
        feats_per_cs = {cs: npz[cs] for cs in COLOR_SPACES}
        print(f'  [{cache_tag}] loaded cached handcrafted features')
        return feats_per_cs

    t0 = time.time()
    feats_per_cs = {cs: [] for cs in COLOR_SPACES}
    for img, mask in zip(images, masks):
        for cs in COLOR_SPACES:
            feats_per_cs[cs].append(extract_features_single_space(img, mask, cs))
    feats_per_cs = {cs: np.array(v, dtype=np.float32) for cs, v in feats_per_cs.items()}
    np.savez(cache_path, **feats_per_cs)
    print(f'  [{cache_tag}] extracted {[f"{cs}:{feats_per_cs[cs].shape}" for cs in COLOR_SPACES]} '
          f'in {time.time()-t0:.1f}s')
    return feats_per_cs


feat_names_per_cs = {cs: build_feature_names(cs) for cs in COLOR_SPACES}
n_feats_per_cs = len(feat_names_per_cs['RGB'])
print(f'Feature dims per color space : {n_feats_per_cs}  (3 channels x (8 stats + {N_HIST_BINS} hist bins))')
print(f'Combined (4 color spaces)    : {n_feats_per_cs * 4}')

X_feats_per_cs = extract_multi_cs_features_for_dataset(X_rgb_filtered, X_masks, 'primary_statistical')
X_feats_combined = np.concatenate([X_feats_per_cs[cs] for cs in COLOR_SPACES], axis=1)
print(f'\nX_feats_combined (Statistical filter): {X_feats_combined.shape}')
for cs in COLOR_SPACES:
    print(f'  {cs:4s}: {X_feats_per_cs[cs].shape}')

## 5 · 5-Fold Stratified Cross-Validation Splits

`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, applied to `y_raw`. These exact fold
assignments are reused for **every** frozen-embedding configuration below (Experiments 1-3), and the
same `(train_idx, val_idx)` pairs also define the leakage-safe boundaries for partial fine-tuning
in Section 16.

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(np.arange(len(y_raw)), y_raw))

print(f'StratifiedKFold - {K_FOLDS} folds over {len(y_raw)} images (CSVS01)')
for fi, (tr, va) in enumerate(fold_splits):
    cnt = np.bincount(y_raw[va], minlength=NUM_CLASSES)
    print(f'  Fold {fi+1}: train={len(tr):5d}  val={len(va):4d}  val class dist={cnt.tolist()}')

## 6 · Pretrained Backbones — Frozen Feature Extractor Definitions

Used for Experiments 1-3 (the frozen re-ablation on CSVS01). Both are ImageNet-pretrained, head
removed, globally pooled:

- **ResNet-50** -> `avgpool` output flattened -> **2048-d**
- **ConvNeXt-Tiny** -> `features` + `avgpool` output flattened -> **768-d**

Only these two backbones are considered anywhere in this notebook.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

backbone_transform = T.Compose([
    T.ToTensor(),
    T.Resize((BACKBONE_SIZE, BACKBONE_SIZE), antialias=True),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
backbone_transform_eval = backbone_transform   # alias used again in Section 15 (deterministic eval)

# Light augmentation for the fine-tuning TRAIN split only (Section 15) — zucchini crops have no
# fixed orientation, so horizontal/vertical flips are label-preserving.
backbone_transform_train_ft = T.Compose([
    T.ToTensor(),
    T.Resize((BACKBONE_SIZE, BACKBONE_SIZE), antialias=True),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ImageEmbeddingDataset(Dataset):
    def __init__(self, rgb_images, transform):
        self.imgs = rgb_images
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        return self.transform(self.imgs[idx])


def build_resnet50_extractor():
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    extractor = nn.Sequential(*list(model.children())[:-1])   # drop fc -> (N,2048,1,1)
    return extractor.to(device).eval()


def build_convnext_tiny_extractor():
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    extractor = nn.Sequential(model.features, model.avgpool)   # -> (N,768,1,1)
    return extractor.to(device).eval()


BACKBONES = {
    'ResNet-50':      {'builder': build_resnet50_extractor,      'embed_dim': 2048},
    'ConvNeXt-Tiny':  {'builder': build_convnext_tiny_extractor, 'embed_dim': 768},
}
print('Backbones registered:', list(BACKBONES.keys()))

## 7 · Frozen Embedding Extraction & Caching

In [ ]:
@torch.no_grad()
def extract_embeddings(extractor, rgb_images, batch_size=BATCH_SIZE):
    ds = ImageEmbeddingDataset(rgb_images, backbone_transform)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    outs = []
    for batch in dl:
        batch = batch.to(device, non_blocking=True)
        feat = extractor(batch)                 # (B, C, 1, 1) or already (B, C)
        feat = feat.flatten(1)                   # (B, C)
        outs.append(feat.cpu().numpy())
    return np.concatenate(outs, axis=0).astype(np.float32)


def get_or_compute_embeddings(backbone_name):
    cache_path = os.path.join(CACHE_DIR, f'{backbone_name.replace(" ", "_")}_embeddings.npy')
    if os.path.exists(cache_path) and not FORCE_RECOMPUTE_EMBEDDINGS:
        emb = np.load(cache_path)
        print(f'  [{backbone_name}] loaded cached embeddings: {emb.shape}')
        return emb, 0.0

    print(f'  [{backbone_name}] extracting embeddings on {device} ...')
    extractor = BACKBONES[backbone_name]['builder']()
    t0 = time.time()
    emb = extract_embeddings(extractor, X_rgb)
    elapsed = time.time() - t0
    np.save(cache_path, emb)
    del extractor
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    print(f'  [{backbone_name}] extracted {emb.shape} in {elapsed:.1f}s')
    return emb, elapsed


embeddings = {}
extraction_times = {}
for name in BACKBONES:
    emb, t_extract = get_or_compute_embeddings(name)
    embeddings[name] = emb
    extraction_times[name] = t_extract

for name, emb in embeddings.items():
    assert emb.shape == (len(X_rgb), BACKBONES[name]['embed_dim']), \
        f'{name} embedding shape mismatch: {emb.shape}'
print('\nAll frozen embeddings extracted and shape-validated OK')

## 8 · Evaluation Metrics & XGBoost Cross-Validated Trainer

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'R2':       r2_score(y_true, y_pred),
        'RMSE':     mean_squared_error(y_true, y_pred) ** 0.5,
        'MAE':      mean_absolute_error(y_true, y_pred),
        'Spearman': spearmanr(y_true, y_pred).correlation,
        'Kendall':  kendalltau(y_true, y_pred).correlation,
    }


XGB_PARAMS = dict(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=SEED,
    tree_method='hist',
    device=XGB_DEVICE,
    n_jobs=-1,
)


def train_xgb_cv(X, y, y_raw_strat, fold_splits, label='model'):
    '''Trains one XGBRegressor per fold (frozen features, identical across folds), returns OOF
    predictions, per-fold metrics, fitted models, and timing.'''
    n = len(y)
    oof_pred = np.zeros(n, dtype=np.float32)
    fold_metrics = []
    fitted_models = []

    t0 = time.time()
    for fi, (tr_idx, va_idx) in enumerate(fold_splits):
        model = xgb.XGBRegressor(**XGB_PARAMS)
        try:
            model.fit(X[tr_idx], y[tr_idx])
        except Exception:
            cpu_params = {**XGB_PARAMS, 'device': 'cpu'}
            model = xgb.XGBRegressor(**cpu_params)
            model.fit(X[tr_idx], y[tr_idx])

        preds = model.predict(X[va_idx])
        oof_pred[va_idx] = preds
        fold_metrics.append(compute_metrics(y[va_idx], preds))
        fitted_models.append(model)
    fit_time = time.time() - t0

    oof_metrics = compute_metrics(y, oof_pred)
    fold_df = pd.DataFrame(fold_metrics)
    print(f'[{label}] fit_time={fit_time:.1f}s  '
          f"OOF R2={oof_metrics['R2']:.4f}  RMSE={oof_metrics['RMSE']:.4f}  "
          f"MAE={oof_metrics['MAE']:.4f}  Spearman={oof_metrics['Spearman']:.4f}  "
          f"Kendall={oof_metrics['Kendall']:.4f}")
    print(f"           per-fold R2 = {fold_df['R2'].mean():.4f} +/- {fold_df['R2'].std():.4f}")

    return {
        'label': label, 'oof_pred': oof_pred, 'oof_metrics': oof_metrics,
        'fold_metrics': fold_df, 'models': fitted_models, 'fit_time': fit_time,
        'n_features': X.shape[1],
    }

## 9 · Experiment 1 — Backbone Comparison (Frozen Embeddings Only, CSVS01)

In [ ]:
results = {}   # master results dict, keyed by experiment label

for name in BACKBONES:
    results[name] = train_xgb_cv(embeddings[name], y, y_raw, fold_splits, label=name)

In [ ]:
exp1_rows = []
for name in BACKBONES:
    r = results[name]
    row = {'Backbone': name, 'Embed dim': BACKBONES[name]['embed_dim'],
           'Extract time (s)': extraction_times[name], 'XGB fit time (s)': r['fit_time']}
    row.update(r['oof_metrics'])
    exp1_rows.append(row)

exp1_table = pd.DataFrame(exp1_rows).set_index('Backbone')
exp1_table = exp1_table[['Embed dim', 'R2', 'RMSE', 'MAE', 'Spearman', 'Kendall',
                          'Extract time (s)', 'XGB fit time (s)']]
print('Experiment 1 — Backbone Comparison on CSVS01 (frozen embeddings only, OOF metrics)')
exp1_table.round(4)

## 10 · Embedding Space Visualization — PCA & UMAP (Frozen Baseline)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
cmap = 'RdYlGn'

for col, name in enumerate(BACKBONES):
    pca = PCA(n_components=2, random_state=SEED)
    proj_pca = pca.fit_transform(embeddings[name])
    ax = axes[0, col]
    sc = ax.scatter(proj_pca[:, 0], proj_pca[:, 1], c=y, cmap=cmap, s=18, alpha=0.75, edgecolor='k', linewidth=0.2)
    ax.set_title(f'{name} — PCA (var explained: {pca.explained_variance_ratio_.sum()*100:.1f}%)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax, label='Score')

    reducer = umap.UMAP(n_components=2, random_state=SEED)
    proj_umap = reducer.fit_transform(embeddings[name])
    ax = axes[1, col]
    sc = ax.scatter(proj_umap[:, 0], proj_umap[:, 1], c=y, cmap=cmap, s=18, alpha=0.75, edgecolor='k', linewidth=0.2)
    ax.set_title(f'{name} — UMAP')
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
    plt.colorbar(sc, ax=ax, label='Score')

plt.suptitle('CSVS01 — Embedding Space Structure by Backbone (Frozen)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); savefig('03_embedding_pca_umap.png'); plt.show()

## 11 · Predicted vs Actual & Residual Plots (Frozen Baseline)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

for col, name in enumerate(BACKBONES):
    r = results[name]
    pred, actual = r['oof_pred'], y

    ax = axes[0, col]
    ax.scatter(actual, pred, alpha=0.5, s=20, color='steelblue', edgecolor='k', linewidth=0.2)
    lims = [min(actual.min(), pred.min()) - 0.3, max(actual.max(), pred.max()) + 0.3]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Ideal (y=x)')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual Score'); ax.set_ylabel('Predicted Score')
    ax.set_title(f"{name} — Predicted vs Actual (R2={r['oof_metrics']['R2']:.3f})")
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, col]
    residuals = pred - actual
    ax.scatter(actual, residuals, alpha=0.5, s=20, color='indianred', edgecolor='k', linewidth=0.2)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_xlabel('Actual Score'); ax.set_ylabel('Residual (pred - actual)')
    ax.set_title(f'{name} — Residuals (mean={residuals.mean():.3f}, std={residuals.std():.3f})')
    ax.grid(alpha=0.3)

plt.tight_layout(); savefig('04_pred_vs_actual_residuals_frozen.png'); plt.show()

## 12 · XGBoost Feature Importance (Frozen Embeddings Only)

In [ ]:
def mean_feature_importance(fitted_models, n_features):
    imps = np.zeros(n_features, dtype=np.float64)
    for m in fitted_models:
        imps += m.feature_importances_
    return imps / len(fitted_models)


fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, name in zip(axes, BACKBONES):
    r = results[name]
    imp = mean_feature_importance(r['models'], r['n_features'])
    top_idx = np.argsort(imp)[::-1][:20]
    ax.barh([f'dim_{i}' for i in top_idx][::-1], imp[top_idx][::-1], color='teal')
    ax.set_title(f'{name} — Top-20 Embedding Dims by Gain')
    ax.set_xlabel('Mean gain importance')

plt.tight_layout(); savefig('05_xgb_feature_importance_embeddings.png'); plt.show()

## 13 · Experiment 2 — Color-Space Ablation, Re-Run on CSVS01

For each backbone, the frozen embedding is fused with each color space individually (RGB, HSV, LAB,
XYZ) and with Combined, all on the Statistical-filtered CSVS01 images — **exactly the v2/v3
methodology, re-run from scratch on the new dataset** rather than assuming LAB still wins.

Each backbone's own best color space is tracked separately in `BEST_CS_FOR[name]` (unlike v2, which
picked a single global winner) — both backbones are carried forward through the rest of this
notebook regardless of which one scores higher.

In [ ]:
CS_VARIANTS = {**{cs: X_feats_per_cs[cs] for cs in COLOR_SPACES}, 'Combined': X_feats_combined}

for name in BACKBONES:
    for cs_name, cs_feats in CS_VARIANTS.items():
        label = f'{name} + {cs_name}'
        fused = np.concatenate([embeddings[name], cs_feats], axis=1)
        results[label] = train_xgb_cv(fused, y, y_raw, fold_splits, label=label)

In [ ]:
cs_rows = []
for name in BACKBONES:
    r0 = results[name]
    cs_rows.append({'Backbone': name, 'Color space': '(none, embeddings only)',
                     'Feature dim': r0['n_features'], **r0['oof_metrics'], 'XGB fit time (s)': r0['fit_time']})
    for cs_name in CS_VARIANTS:
        r = results[f'{name} + {cs_name}']
        cs_rows.append({'Backbone': name, 'Color space': cs_name,
                         'Feature dim': r['n_features'], **r['oof_metrics'], 'XGB fit time (s)': r['fit_time']})

cs_table = pd.DataFrame(cs_rows)
cs_table = cs_table[['Backbone', 'Color space', 'Feature dim', 'R2', 'RMSE', 'MAE',
                      'Spearman', 'Kendall', 'XGB fit time (s)']]
print('Experiment 2 — Color-Space Ablation on CSVS01 (OOF metrics)')
cs_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
cs_order = ['(none, embeddings only)'] + list(CS_VARIANTS.keys())
colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(cs_order)))

for ax, name in zip(axes, BACKBONES):
    r2_vals = [cs_table[(cs_table['Backbone'] == name) & (cs_table['Color space'] == cs)]['R2'].values[0]
               for cs in cs_order]
    bars = ax.bar(cs_order, r2_vals, color=colors, edgecolor='black')
    ax.set_title(f'{name} — R2 by Color Space Fused with Embeddings (CSVS01)')
    ax.set_ylabel('OOF R2'); ax.tick_params(axis='x', rotation=30)
    for b, v in zip(bars, r2_vals):
        ax.text(b.get_x() + b.get_width()/2, v, f'{v:.3f}', ha='center',
                va='bottom' if v >= 0 else 'top', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Experiment 2 — Color-Space Ablation, CSVS01 (Statistical-filtered)', y=1.03)
plt.tight_layout(); savefig('06_colorspace_ablation_r2.png'); plt.show()

BEST_CS_FOR = {}
for name in BACKBONES:
    sub = cs_table[cs_table['Backbone'] == name]
    best_row = sub.loc[sub['R2'].idxmax()]
    BEST_CS_FOR[name] = best_row['Color space']
    print(f"Best color-space for {name}: {best_row['Color space']}  (R2={best_row['R2']:.4f})")

## 14 · Experiment 3 — Filtering-Method Ablation, Per Backbone, Re-Run on CSVS01

v2 ran the filtering ablation once, on a single globally-best backbone+color-space combination.
Since **both** backbones are being carried forward here, this ablation is run **independently for
each backbone**, using that backbone's own best color space from Section 13. `BEST_FILTER_FOR[name]`
tracks the winner for each backbone separately.

In [ ]:
filter_feat_cache = {PRIMARY_FILTER: X_feats_per_cs}
for fname in FILTER_METHODS:
    if fname == PRIMARY_FILTER:
        continue
    tag = fname.replace(' ', '_').replace('(', '').replace(')', '').replace('=', '').replace('|', '').replace('<', 'lt')
    filter_feat_cache[fname] = extract_multi_cs_features_for_dataset(
        filter_outputs[fname]['images'], filter_outputs[fname]['masks'], tag)


def features_for_filter_and_cs(fname, cs_name):
    fc = filter_feat_cache[fname]
    if cs_name == 'Combined':
        return np.concatenate([fc[cs] for cs in COLOR_SPACES], axis=1)
    return fc[cs_name]


FILTER_TABLES = {}
BEST_FILTER_FOR = {}

for name in BACKBONES:
    cs_name = BEST_CS_FOR[name]
    print(f'\nFiltering ablation for {name}  (color space = {cs_name})')
    rows = []
    for fname in FILTER_METHODS:
        label = f'{name} + {cs_name} [{fname}]'
        fused = np.concatenate([embeddings[name], features_for_filter_and_cs(fname, cs_name)], axis=1)
        results[label] = train_xgb_cv(fused, y, y_raw, fold_splits, label=label)
        r = results[label]
        rows.append({'Filter method': fname, 'Feature dim': r['n_features'],
                     **r['oof_metrics'], 'XGB fit time (s)': r['fit_time']})
    table = pd.DataFrame(rows).set_index('Filter method')
    table = table[['Feature dim', 'R2', 'RMSE', 'MAE', 'Spearman', 'Kendall', 'XGB fit time (s)']]
    FILTER_TABLES[name] = table
    best_filter = table['R2'].idxmax()
    BEST_FILTER_FOR[name] = best_filter
    print(f'  Best filter for {name}: {best_filter}  (R2={table.loc[best_filter, "R2"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
metrics_to_plot = ['R2', 'Spearman', 'Kendall']
for ax, name in zip(axes, BACKBONES):
    table = FILTER_TABLES[name]
    x = np.arange(len(FILTER_METHODS)); width = 0.25
    for i, metric in enumerate(metrics_to_plot):
        ax.bar(x + (i - 1) * width, table[metric].values, width, label=metric)
    ax.set_xticks(x); ax.set_xticklabels(list(FILTER_METHODS.keys()), rotation=15, ha='right')
    ax.set_ylabel('Score'); ax.set_title(f'{name} + {BEST_CS_FOR[name]} — Filtering-Method Ablation')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); savefig('07_filtering_ablation_per_backbone.png'); plt.show()

for name in BACKBONES:
    print(f'{name}: best color space = {BEST_CS_FOR[name]}, best filter = {BEST_FILTER_FOR[name]}  '
          f'(R2={FILTER_TABLES[name].loc[BEST_FILTER_FOR[name], "R2"]:.4f})')

## 15 · Partial Fine-Tuning — Setup and Leakage-Safe Protocol

v2's backbones are **100% frozen** — no gradient ever flows into the CNN. Here, the **last block**
of each backbone is unfrozen and fine-tuned:

- **ResNet-50** -> unfreeze `layer4` (final residual stage)
- **ConvNeXt-Tiny** -> unfreeze `features[-1]` (final ConvNeXt stage)

Everything upstream of that block stays frozen at ImageNet weights.

**Why fine-tuning needs its own protocol, not just a flag:** XGBoost is not differentiable, so the
backbone cannot be fine-tuned directly against the XGBoost loss. Instead, a **temporary trainable
regression head** (`RegressionHead`, a small 2-layer MLP) is attached during fine-tuning only, and
discarded afterward — only the adapted backbone weights are kept, to re-extract embeddings that then
feed XGBoost exactly as in Sections 7-14.

**Leakage-safe CV protocol (Section 16):** if the backbone were fine-tuned once on the *entire*
dataset before running 5-fold CV, every fold's "held-out" validation images would already have
influenced the backbone weights used to embed them — an optimistic, invalid R². To avoid this, the
backbone is fine-tuned **independently inside every fold**:

1. Take that fold's `train_idx` (never `val_idx`).
2. Carve out `FT_INNER_VAL_FRAC` = 15% of `train_idx` (stratified by score) as an **inner**
   validation set, used only to early-stop the fine-tuning itself.
3. Fine-tune the unfrozen block + temporary head on the remaining ~85% of `train_idx`, restoring the
   backbone weights from the best inner-validation epoch.
4. Discard the temporary head. Extract embeddings for the fold's full `train_idx` and `val_idx` using
   this fold's own fine-tuned backbone.
5. Fit XGBoost on `train_idx` embeddings + handcrafted features, predict `val_idx` — exactly as
   Sections 8-14.

`val_idx` for that fold is *never* seen by anything (backbone or head) until the final XGBoost
prediction step. Because backbone weights are re-initialized from ImageNet at the start of every
fold, this trains **5 independent partially fine-tuned backbones per model** (10 total across both
backbones) — meaningfully slower than the frozen pipeline, but the only way to get a trustworthy OOF
R² for a fine-tuned model.

Each backbone's fine-tuning uses its own Section 13/14 winning color-space + filter combination
(`BEST_CS_FOR[name]`, `BEST_FILTER_FOR[name]`) for the handcrafted-feature half of the fused vector.

In [ ]:
class RegressionHead(nn.Module):
    '''Temporary head used only during fine-tuning; discarded before extracting embeddings.'''
    def __init__(self, in_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class ResNet50FTBackbone(nn.Module):
    '''ResNet-50 with everything frozen except layer4 (final residual stage).'''
    def __init__(self):
        super().__init__()
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = m.layer1, m.layer2, m.layer3, m.layer4
        self.avgpool = m.avgpool
        for p in self.parameters():
            p.requires_grad = False

    def unfreeze_last_block(self):
        for p in self.layer4.parameters():
            p.requires_grad = True

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)


class ConvNeXtTinyFTBackbone(nn.Module):
    '''ConvNeXt-Tiny with everything frozen except features[-1] (final stage).'''
    def __init__(self):
        super().__init__()
        m = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        self.features = m.features
        self.avgpool = m.avgpool
        for p in self.parameters():
            p.requires_grad = False

    def unfreeze_last_block(self):
        for p in self.features[-1].parameters():
            p.requires_grad = True

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)


FT_BACKBONE_BUILDERS = {
    'ResNet-50':      ResNet50FTBackbone,
    'ConvNeXt-Tiny':  ConvNeXtTinyFTBackbone,
}


class FTImageDataset(Dataset):
    def __init__(self, rgb_images, targets, transform):
        self.imgs = rgb_images
        self.targets = targets.astype(np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        return self.transform(self.imgs[idx]), self.targets[idx]


def finetune_backbone_one_fold(backbone_name, X_rgb_source, y_cont, inner_tr_idx, inner_val_idx,
                                epochs=FT_EPOCHS, patience=FT_PATIENCE, batch_size=FT_BATCH_SIZE):
    '''Fine-tunes the last block of `backbone_name` on inner_tr_idx only, early-stopping on
    inner_val_idx. Returns the backbone (best-epoch weights, temporary head discarded) + history.'''
    backbone = FT_BACKBONE_BUILDERS[backbone_name]().to(device)
    backbone.unfreeze_last_block()
    head = RegressionHead(BACKBONES[backbone_name]['embed_dim']).to(device)

    trainable_backbone_params = [p for p in backbone.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': trainable_backbone_params, 'lr': FT_LR_BACKBONE},
        {'params': head.parameters(),         'lr': FT_LR_HEAD},
    ], weight_decay=FT_WEIGHT_DECAY)
    criterion = nn.MSELoss()

    train_dl = DataLoader(
        FTImageDataset(X_rgb_source[inner_tr_idx], y_cont[inner_tr_idx], backbone_transform_train_ft),
        batch_size=batch_size, shuffle=True, num_workers=0)
    val_dl = DataLoader(
        FTImageDataset(X_rgb_source[inner_val_idx], y_cont[inner_val_idx], backbone_transform_eval),
        batch_size=batch_size, shuffle=False, num_workers=0)

    best_val_mse, best_epoch, bad_epochs = float('inf'), 0, 0
    best_state = None
    history = {'train_mse': [], 'val_mse': []}

    for epoch in range(1, epochs + 1):
        backbone.train(); head.train()
        train_losses = []
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = head(backbone(xb))
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        backbone.eval(); head.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = head(backbone(xb))
                val_losses.append(criterion(pred, yb).item())

        train_mse, val_mse = float(np.mean(train_losses)), float(np.mean(val_losses))
        history['train_mse'].append(train_mse); history['val_mse'].append(val_mse)

        if val_mse < best_val_mse - 1e-4:
            best_val_mse, best_epoch, bad_epochs = val_mse, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in backbone.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    if best_state is not None:
        backbone.load_state_dict(best_state)
    backbone.eval()
    history['best_epoch'] = best_epoch
    history['best_val_mse'] = best_val_mse
    return backbone, history


print('Fine-tuning building blocks ready: RegressionHead, ResNet50FTBackbone, '
      'ConvNeXtTinyFTBackbone, FTImageDataset, finetune_backbone_one_fold()')

## 16 · Experiment 4 — Per-Fold Partial Fine-Tuning + XGBoost, 5-Fold CV

Runs the full leakage-safe protocol from Section 15 for both backbones, using each backbone's own
Section 13/14 winning color-space + filter combination. **This is the slowest section in the
notebook** (2 backbones x 5 folds = 10 independent fine-tune runs, each up to `FT_EPOCHS` epochs).

In [ ]:
def train_finetuned_cv(backbone_name, cs_feats, fold_splits, y, y_raw_strat, X_rgb_source, label):
    n = len(y)
    oof_pred = np.zeros(n, dtype=np.float32)
    oof_embeddings = np.zeros((n, BACKBONES[backbone_name]['embed_dim']), dtype=np.float32)
    fold_metrics = []
    ft_histories = []
    fitted_models = []

    t0 = time.time()
    for fi, (tr_idx, va_idx) in enumerate(fold_splits):
        inner_tr_idx, inner_val_idx = train_test_split(
            tr_idx, test_size=FT_INNER_VAL_FRAC, random_state=SEED, stratify=y_raw_strat[tr_idx])

        backbone, hist = finetune_backbone_one_fold(
            backbone_name, X_rgb_source, y, inner_tr_idx, inner_val_idx)
        ft_histories.append(hist)

        emb_tr = extract_embeddings(backbone, X_rgb_source[tr_idx])
        emb_va = extract_embeddings(backbone, X_rgb_source[va_idx])
        oof_embeddings[va_idx] = emb_va   # OOF-safe: only ever from a backbone that never saw va_idx

        X_tr = np.concatenate([emb_tr, cs_feats[tr_idx]], axis=1)
        X_va = np.concatenate([emb_va, cs_feats[va_idx]], axis=1)

        model = xgb.XGBRegressor(**XGB_PARAMS)
        try:
            model.fit(X_tr, y[tr_idx])
        except Exception:
            cpu_params = {**XGB_PARAMS, 'device': 'cpu'}
            model = xgb.XGBRegressor(**cpu_params)
            model.fit(X_tr, y[tr_idx])

        preds = model.predict(X_va)
        oof_pred[va_idx] = preds
        fold_metrics.append(compute_metrics(y[va_idx], preds))
        fitted_models.append(model)

        del backbone
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        print(f"  [{label}] fold {fi+1}/{K_FOLDS} done — val R2={fold_metrics[-1]['R2']:.4f}  "
              f"best_ft_epoch={hist['best_epoch']}  inner_val_mse={hist['best_val_mse']:.4f}")

    fit_time = time.time() - t0
    oof_metrics = compute_metrics(y, oof_pred)
    fold_df = pd.DataFrame(fold_metrics)
    print(f'[{label}] TOTAL time={fit_time/60:.1f}min  '
          f"OOF R2={oof_metrics['R2']:.4f}  RMSE={oof_metrics['RMSE']:.4f}  "
          f"MAE={oof_metrics['MAE']:.4f}  Spearman={oof_metrics['Spearman']:.4f}  "
          f"Kendall={oof_metrics['Kendall']:.4f}")
    print(f"           per-fold R2 = {fold_df['R2'].mean():.4f} +/- {fold_df['R2'].std():.4f}")

    return {
        'label': label, 'oof_pred': oof_pred, 'oof_embeddings': oof_embeddings,
        'oof_metrics': oof_metrics, 'fold_metrics': fold_df, 'models': fitted_models,
        'ft_histories': ft_histories, 'fit_time': fit_time,
    }


finetune_results = {}
for name in BACKBONES:
    cs_name, fname = BEST_CS_FOR[name], BEST_FILTER_FOR[name]
    cs_feats = features_for_filter_and_cs(fname, cs_name)
    label = f'{name} + {cs_name} [{fname}] (fine-tuned)'
    print(f'\n{"="*80}\nFine-tuning {name}  (color space={cs_name}, filter={fname})\n{"="*80}')
    finetune_results[name] = train_finetuned_cv(
        name, cs_feats, fold_splits, y, y_raw, X_rgb, label=label)

## 17 · Frozen vs Fine-Tuned Comparison

In [ ]:
compare_rows = []
for name in BACKBONES:
    frozen_label = f'{name} + {BEST_CS_FOR[name]} [{BEST_FILTER_FOR[name]}]'
    frozen_r2 = results[frozen_label]['oof_metrics']['R2']
    ft_r2 = finetune_results[name]['oof_metrics']['R2']
    compare_rows.append({
        'Backbone': name, 'Color space': BEST_CS_FOR[name], 'Filter': BEST_FILTER_FOR[name],
        'Frozen R2': frozen_r2, 'Fine-tuned R2': ft_r2, 'Delta R2': ft_r2 - frozen_r2,
        'Frozen RMSE': results[frozen_label]['oof_metrics']['RMSE'],
        'Fine-tuned RMSE': finetune_results[name]['oof_metrics']['RMSE'],
    })
compare_table = pd.DataFrame(compare_rows).set_index('Backbone')
print('Frozen (Section 14) vs Partially Fine-Tuned (Section 16) — best color-space+filter combo per backbone')
compare_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(BACKBONES)); width = 0.35
frozen_vals = [compare_table.loc[n, 'Frozen R2'] for n in BACKBONES]
ft_vals = [compare_table.loc[n, 'Fine-tuned R2'] for n in BACKBONES]
b1 = ax.bar(x - width/2, frozen_vals, width, label='Frozen backbone', color='steelblue')
b2 = ax.bar(x + width/2, ft_vals, width, label='Partially fine-tuned', color='darkorange')
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{b.get_height():.3f}',
                ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(list(BACKBONES.keys()))
ax.set_ylabel('OOF R2'); ax.set_title('Frozen vs Partially Fine-Tuned — Best Config per Backbone (CSVS01)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); savefig('08_frozen_vs_finetuned_r2.png'); plt.show()

## 18 · Confusion Matrix (Fine-Tuned Models, Rounded OOF Predictions)

Continuous OOF predictions are rounded to the nearest integer and clipped to `[1, 9]` to build a
standard true-vs-predicted confusion matrix, one per backbone (using each backbone's fine-tuned
model from Section 16).

In [ ]:
def scores_to_int(vals):
    return np.clip(np.round(vals), 1, 9).astype(int)

y_true_int = scores_to_int(y)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, name in zip(axes, BACKBONES):
    r = finetune_results[name]
    pred_int = scores_to_int(r['oof_pred'])
    cm = confusion_matrix(y_true_int, pred_int, labels=list(range(1, 10)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=range(1, 10), yticklabels=range(1, 10), cbar=False)
    ax.set_xlabel('Predicted score'); ax.set_ylabel('True score')
    ax.set_title(f"{name} — Confusion Matrix (fine-tuned, OOF, R2={r['oof_metrics']['R2']:.3f})")

plt.tight_layout(); savefig('09_confusion_matrix.png'); plt.show()

## 19 · Confusion Frieze — Image Strip per True Score

For each true score (one row = one score, 1-9), up to `FRIEZE_MAX_PER_ROW` (10) example crops are
shown, evenly sampled across that row's predictions **sorted ascending by predicted score** — so a
row with a wide color spread visually "drifts" left-to-right, and a tight, well-clustered row shows
the model is confident and correct for that score. Each thumbnail's border is colored by rounded
prediction error: **green** = exact match, **orange** = off by 1, **red** = off by 2 or more.

In [ ]:
def build_confusion_frieze(name, r, X_rgb_source, y_true_int_arr, max_per_row=FRIEZE_MAX_PER_ROW):
    pred_cont = r['oof_pred']
    pred_int = scores_to_int(pred_cont)

    fig, axes = plt.subplots(9, max_per_row, figsize=(max_per_row * 1.7, 9 * 1.85))
    for row, true_score in enumerate(range(1, 10)):
        idx = np.where(y_true_int_arr == true_score)[0]
        order = idx[np.argsort(pred_cont[idx])]
        if len(order) > max_per_row:
            sel = order[np.linspace(0, len(order) - 1, max_per_row).astype(int)]
        else:
            sel = order

        for col in range(max_per_row):
            ax = axes[row, col]
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if col < len(sel):
                i = sel[col]
                img = X_rgb_source[i]
                ax.imshow(img)
                err = int(pred_int[i]) - true_score
                color = 'green' if err == 0 else ('orange' if abs(err) == 1 else 'red')
                h, w = img.shape[:2]
                ax.add_patch(Rectangle((0, 0), w - 1, h - 1, linewidth=4,
                                        edgecolor=color, facecolor='none'))
                ax.set_title(f'pred={pred_cont[i]:.1f}', fontsize=8, color=color)
            else:
                ax.axis('off')
        axes[row, 0].set_ylabel(f'True={true_score}', fontsize=11, rotation=0,
                                 labelpad=38, va='center')

    plt.suptitle(f'{name} — Confusion Frieze (fine-tuned, OOF predictions, sorted by prediction)',
                 y=1.001, fontsize=14, fontweight='bold')
    plt.tight_layout()
    savefig(f'10_confusion_frieze_{name.replace(" ", "_").replace("-", "")}.png', dpi=130)
    plt.show()


for name in BACKBONES:
    build_confusion_frieze(name, finetune_results[name], X_rgb, y_true_int)

## 20 · Final Master Comparison Table

Every configuration run in this notebook — frozen backbone-only, color-space ablation, per-backbone
filter ablation, and partial fine-tuning — side by side, sorted by R².

In [ ]:
master_rows = []

for name in BACKBONES:
    r = results[name]
    master_rows.append({
        'Experiment': 'Exp 1: Frozen backbone only', 'Configuration': name, 'Backbone': name,
        'Color space': '(none)', 'Filter': '-', 'Feature dim': r['n_features'],
        'Extract/FT time (s)': round(extraction_times[name], 2),
        'XGB fit time (s)': round(r['fit_time'], 2),
        **{k: round(v, 4) for k, v in r['oof_metrics'].items()},
    })
    for cs_name in CS_VARIANTS:
        label = f'{name} + {cs_name}'
        r = results[label]
        master_rows.append({
            'Experiment': 'Exp 2: Color-space ablation', 'Configuration': label, 'Backbone': name,
            'Color space': cs_name, 'Filter': PRIMARY_FILTER, 'Feature dim': r['n_features'],
            'Extract/FT time (s)': round(extraction_times[name], 2),
            'XGB fit time (s)': round(r['fit_time'], 2),
            **{k: round(v, 4) for k, v in r['oof_metrics'].items()},
        })
    for fname in FILTER_METHODS:
        label = f'{name} + {BEST_CS_FOR[name]} [{fname}]'
        r = results[label]
        master_rows.append({
            'Experiment': 'Exp 3: Filtering ablation', 'Configuration': label, 'Backbone': name,
            'Color space': BEST_CS_FOR[name], 'Filter': fname, 'Feature dim': r['n_features'],
            'Extract/FT time (s)': round(extraction_times[name], 2),
            'XGB fit time (s)': round(r['fit_time'], 2),
            **{k: round(v, 4) for k, v in r['oof_metrics'].items()},
        })
    r = finetune_results[name]
    master_rows.append({
        'Experiment': 'Exp 4: Partial fine-tuning', 'Configuration': r['label'], 'Backbone': name,
        'Color space': BEST_CS_FOR[name], 'Filter': BEST_FILTER_FOR[name], 'Feature dim': '-',
        'Extract/FT time (s)': round(r['fit_time'], 2), 'XGB fit time (s)': '-',
        **{k: round(v, 4) for k, v in r['oof_metrics'].items()},
    })

master_table = pd.DataFrame(master_rows)
master_table = master_table[['Experiment', 'Configuration', 'Backbone', 'Color space', 'Filter',
                              'Feature dim', 'R2', 'RMSE', 'MAE', 'Spearman', 'Kendall',
                              'Extract/FT time (s)', 'XGB fit time (s)']]
master_table = master_table.sort_values('R2', ascending=False).reset_index(drop=True)

master_table.to_csv(os.path.join(OUT_DIR, 'master_comparison_table.csv'), index=False)
print(f"Saved -> {os.path.join(OUT_DIR, 'master_comparison_table.csv')}")
print(f"Total configurations benchmarked: {len(master_table)}\n")

(master_table.style
 .background_gradient(subset=['R2', 'Spearman', 'Kendall'], cmap='Greens')
 .background_gradient(subset=['RMSE', 'MAE'], cmap='Reds_r')
 .format({'R2': '{:.4f}', 'RMSE': '{:.4f}', 'MAE': '{:.4f}',
          'Spearman': '{:.4f}', 'Kendall': '{:.4f}'}))

## 21 · Summary & Key Takeaways

In [ ]:
best_row = master_table.iloc[0]

print('=' * 88)
print('SUMMARY — LAB Winners Deep Ablation (CSVS01, Partial Fine-Tuning, Confusion Diagnostics)')
print('=' * 88)
print(f"\nBest overall configuration : {best_row['Configuration']}")
print(f"  R2={best_row['R2']:.4f}  RMSE={best_row['RMSE']:.4f}  MAE={best_row['MAE']:.4f}  "
      f"Spearman={best_row['Spearman']:.4f}  Kendall={best_row['Kendall']:.4f}")

print('\nBackbone comparison (frozen embeddings only, CSVS01):')
for name in BACKBONES:
    m = results[name]['oof_metrics']
    print(f"  {name:16s} R2={m['R2']:.4f}  RMSE={m['RMSE']:.4f}  dim={BACKBONES[name]['embed_dim']:5d}")

print('\nColor-space + filter ablation winners (per backbone):')
for name in BACKBONES:
    print(f"  {name}: best color space = {BEST_CS_FOR[name]}, best filter = {BEST_FILTER_FOR[name]}  "
          f"(R2={FILTER_TABLES[name].loc[BEST_FILTER_FOR[name], 'R2']:.4f})")

print('\nFrozen vs partially fine-tuned (best color-space+filter combo per backbone):')
for name in BACKBONES:
    row = compare_table.loc[name]
    print(f"  {name}: frozen R2={row['Frozen R2']:.4f}  ->  fine-tuned R2={row['Fine-tuned R2']:.4f}  "
          f"(delta={row['Delta R2']:+.4f})")

print('\nAll figures saved to:', os.path.abspath(OUT_DIR))
print('All cached embeddings/features saved to:', os.path.abspath(CACHE_DIR))
print('=' * 88)

### Notes for interpretation

- **OOF (out-of-fold) metrics** throughout Sections 9-14 come from the identical 5-fold
  `StratifiedKFold` split; **Section 16's fine-tuned OOF metrics** come from 5 independently
  fine-tuned backbones (one per fold) predicting only on data they never trained or early-stopped
  on — both are unbiased generalization estimates, but note the fine-tuned run is more
  computationally expensive per point of R² gained (see `Extract/FT time (s)` in the master table).
- **Color-space (Exp 2) and filter (Exp 3) ablation were re-run from scratch on CSVS01** rather than
  assuming v2's LAB + Statistical winners still hold — check `BEST_CS_FOR` / `BEST_FILTER_FOR` above
  against v2's LAB + Statistical result before treating the two datasets as directly comparable.
- **Confusion matrix vs confusion frieze**: the matrix (Section 18) gives the aggregate picture —
  which true/predicted score pairs are most common — while the frieze (Section 19) lets you eyeball
  *which specific images* are driving the errors in a given row, and whether a score's errors are a
  tight off-by-one drift or scattered/noisy.
- If fine-tuning's R² gain over frozen embeddings is small relative to its ~10x extra compute cost
  (10 fine-tune runs vs cached frozen embeddings), that's a legitimate production-tradeoff finding,
  not a failure of the experiment — report it either way.
- To change which backbone block is fine-tuned, edit `unfreeze_last_block()` in
  `ResNet50FTBackbone` / `ConvNeXtTinyFTBackbone` (Section 15) — e.g. also unfreeze `layer3` /
  `features[5]` for a deeper (but slower, more overfit-prone) fine-tune.